# 05 – Mechanistic results in the PINN format

The Mechanistic model is calibrated (ABC-SMC) in its own repository. Here its raw results are converted into the same files that the PINN produces, so that tables and plots (notebooks 06 and 07) read the two models in the same way:

| File | Same as the PINN | Content |
|---|---|---|
| `abc_distributions.pkl` | `mcmc_distributions.pkl` | posterior samples of every week: `R0s_samples`, `Deltas_samples` (or `delta_fixed`), `inf_t0s_samples`, `Rts_samples`, `weeks` |
| `forecast_h<horizon>/forecast_results.pickle` | `forecast_h<horizon>/forecast_results.pickle` | forecast of every week: `raw`, `summary_90`, `summary_50`, `quantiles`, `t0`, `date`, `y_obs` |

One notebook for the five cases, chosen with `CASE` as in notebooks 03 and 04. The two kinds of case differ in the parameters:

| `CASE` | Parameters in `posterior.csv` | Saved |
|---|---|---|
| `"real"` | R0, Delta, delta_t, inf_t0 | Delta samples; `delta_t` under `tau0s_samples` |
| `"synthetic"` | R0, inf_t0 (Delta is fixed to 21) | `delta_fixed`; no `tau0s_samples` |

`delta_t` is stored under `tau0s_samples` only so that the key exists as for the PINN: it is **not** the same quantity as the onset tau0 of the PINN (the Mechanistic model estimates the initial number of infected, `inf_t0`, while the PINN fixes it to one). R0 and Delta are directly comparable.

**Input.** The raw results of the Mechanistic repository, one folder per case (`RAW_DIR`), with one subfolder per look-back (`nall` = all the history, `n4`, `n5`, `n6` = last weeks), each with one folder per week with `posterior.csv`, `Rt.npz`, `trajectories.npz` (and `weights.npz` if the particles are resampled). For the synthetic scenarios the extra fit at the last week (`retro_w34` for R0 = 1.5, `retro_w24` for R0 = 2.0, with `posterior.csv`, `trajectories.npz` and `infections_by_age.npz`) is read too and saved as `last_week` in `abc_distributions.pkl`: the samples of R0 are used for the density of R0 in notebook 07, the trajectories (total and by age class) for the retrospective fit and the age sanity check in notebook 08. They are not part of this repository.

**Output.** `outputs/posteriors/mechanistic/<case>/`, with the same case names as the PINN (`real_London_lookback25` for `nall`, `real_London_lookback5` for `n5`, `synthetic_R0_1.5_noise_light` for all the history). The Mechanistic model has no seeds. The window of the forecast starts at the first week of the look-back, as for the PINN. To use the files as the results of the paper, copy them to `precomputed/posteriors/mechanistic/`.

No plots here.

In [ ]:
import os

from pinnvsmech.inference.cases import load_case
from pinnvsmech.mechanistic.convert import convert_variant, find_variants, save_converted, variant_dt_train
from pinnvsmech.paths import DATA_DIR, OUTPUTS_DIR

## Settings

In [ ]:
CASE = "synthetic"                 # "synthetic" or "real"
R0_TRUE = 1.5                      # synthetic only: 1.5 or 2.0
NOISE = "light"                    # synthetic only: "light" or "mild"

# Raw results of the Mechanistic model for this case: a folder with the look-back folders (nall, n5, ...) inside
RAW_DIR = DATA_DIR / "mechanistic_raw" / ("real_London" if CASE == "real" else f"synthetic_R0_{R0_TRUE}_noise_{NOISE}")
VARIANTS = None                    # None: all the look-back folders found in RAW_DIR; or e.g. ["nall", "n5"]

HORIZONS = [4]                     # weeks ahead: 4 (short-term), 10 (long-term) or both [4, 10]
RESAMPLE = False                   # True: resample the particles with the ABC-SMC weights (needs weights.npz)
N_RESAMPLE = 1000                  # particles after the resampling
SEED = 0                           # seed of the resampling

OUT_DIR = OUTPUTS_DIR / "posteriors" / "mechanistic"

## Conversion

In [ ]:
# Observations of the case (dates of the weeks, weekly data); the look-back is set below for every variant
case = load_case(CASE, R0=R0_TRUE, noise=NOISE)
n_weeks = len(case.obs)

variants = find_variants(RAW_DIR) if VARIANTS is None else VARIANTS
print("variants:", variants)

for variant in variants:
    distributions, forecasts = convert_variant(
        raw_dir=RAW_DIR, variant=variant, dates=case.dates, obs=case.obs, horizons=HORIZONS,
        delta_fixed=case.delta_fixed, resample=RESAMPLE, n_resample=N_RESAMPLE, seed=SEED,
    )

    # Same name of the case as the PINN, with the look-back of the variant
    dt_train = variant_dt_train(variant, n_weeks)
    label = load_case(CASE, R0=R0_TRUE, noise=NOISE, dt_train=None if dt_train >= n_weeks else dt_train).label

    save_converted(OUT_DIR / label, distributions, forecasts)
    print("->", OUT_DIR / label)

# PINN t0_range, to see which weeks the Mechanistic model has or lacks
weeks_mech = set(distributions["weeks"])
print("weeks of the PINN not in the Mechanistic results:", sorted(set(case.t0_range) - weeks_mech))